# Notebook 5 — Concept Bottleneck Multi-task Model

**Amaç:** Pretrained ViT-L encoder + 4 head birleşimini tek bir modelde topla. Mimari, weight loading, forward pass smoke test, parameter count, memory profile burada yapılır. Asıl training Notebook 6'da.

## Mimari

```
                    Input (B, 1, 64, 64, 64)
                            │
                  ┌─────────▼─────────┐
                  │  ViT-Large encoder │  ← DAPT epoch 3 ckpt'tan yüklenir
                  │  patch 16³, depth 24│
                  └─────────┬─────────┘
                            │ tokens (B, 64, 1024)
              ┌─────────────┼───────────────┐
              │             │               │
        global pool    intermediate    intermediate
        (B, 1024)      layer 6,12,18   layer 6,12,18
              │             │               │
       ┌──────┴──────┐      │               │
       │             │      ▼               ▼
       ▼             ▼  UNETR-lite     UNETR-lite
   Detection    Concept  decoder      decoder
     MLP         MLP    (skip conn.) (skip conn.)
       │           │         │
       │           ▼         │
       │      Concept (B,8)  │
       │           │         │
       │     ┌─────▼─────┐   │
       │     │ Linear 8→2 │   │
       │     │ (BOTTLENECK)│  │ ← Concept Bottleneck:
       │     └─────┬─────┘   │   malignancy SADECE
       │           │         │   concept'lerden tahmin edilir
       ▼           ▼         ▼
   Detection  Malignancy Segmentation
   (B, 2)     (B, 2)     (B, 1, 64³)
```

## 4 head, 4 task

| Head | Output | Loss | Etkin sample'lar |
|---|---|---|---|
| **Detection** | (B, 2) binary | CrossEntropy | Hepsi |
| **Concepts** | (B, 8) regresyon | MSE (mask: -1 → ignore) | Sadece pozitif |
| **Malignancy** | (B, 2) binary | CrossEntropy (ignore_index=-1) | Sadece pozitif + label∈{0,1} |
| **Segmentation** | (B, 1, 64³) | Dice + BCE (mask: seg_loss_mask=0 → ignore) | Pozitif + mask'li |

**Concept Bottleneck:** Malignancy head'in input'u **doğrudan encoder feature'ı değil**, sadece 8 concept değeridir. Yani malignancy kararı **mecburen klinik kavramlar üzerinden** açıklanır. Bu CBM (Koh et al. 2020) standart formu.

## Pipeline akışı

1. Setup (imports, paths)
2. UNETR-lite segmentation decoder class
3. CBM Multi-task model class
4. Model oluştur + DAPT encoder yükle
5. Parameter count
6. Forward pass smoke test (random input)
7. Loss function tanımları
8. Loss smoke test (dummy batch)
9. Memory profile (batch boyutları için VRAM)
10. Özet

**Süre:** ~10-15 dakika (Drive'dan checkpoint indirme dahil)


## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Mounted at /content/drive
PyTorch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


## 2. MONAI yükle (ViT için)

In [2]:
!pip install monai --quiet
from monai.networks.nets import ViT
print('MONAI yüklendi')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 69.8 MB/s eta 0:00:00


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


MONAI yüklendi


## 3. Yollar ve model config

In [3]:
# Pretrained checkpoint (DAPT epoch 3)
DAPT_CKPT = '/content/drive/MyDrive/bitirme/Checkpoints_DAPT/pulmomae_dapt_epoch3_20260525-210449_model_only.pth'

# Encoder config — pretrain kodundan birebir
ROI_SIZE     = (64, 64, 64)
PATCH_SIZE   = (16, 16, 16)
HIDDEN_SIZE  = 1024
MLP_DIM      = 4096
NUM_LAYERS   = 24
NUM_HEADS    = 16
N_PATCHES_PER_DIM = ROI_SIZE[0] // PATCH_SIZE[0]   # 4
N_TOKENS     = N_PATCHES_PER_DIM ** 3              # 64

# Multi-task config
N_CONCEPTS = 8
SKIP_LAYERS = (6, 12, 18)  # UNETR-lite skip connections (1-indexed)

print(f'Encoder: ViT-L, {NUM_LAYERS} layer, {HIDDEN_SIZE} hidden')
print(f'Tokens: {N_PATCHES_PER_DIM}³ = {N_TOKENS} per sample')
print(f'Concepts: {N_CONCEPTS}')
print(f'Skip layers: {SKIP_LAYERS}')

Encoder: ViT-L, 24 layer, 1024 hidden
Tokens: 4³ = 64 per sample
Concepts: 8
Skip layers: (6, 12, 18)


## 4. UNETR-lite Segmentation Decoder

ViT-L encoder'ın 3 ara katmanını (layer 6, 12, 18) skip connection olarak kullanır. Bottleneck'ten 4 stage upsample ile (4³ → 8³ → 16³ → 32³ → 64³) tam çözünürlüğe çıkar.

In [4]:
class UNETRLiteDecoder(nn.Module):
    """
    UNETR-inspired 3D segmentation decoder.

    Input:
      tokens: (B, N=64, D=1024) - encoder final output
      hidden_states: list of (B, N=64, D=1024) per encoder layer

    Output:
      seg_logits: (B, 1, 64, 64, 64)
    """

    def __init__(self, encoder_dim=1024, n_per_dim=4, skip_layers=(6, 12, 18)):
        super().__init__()
        self.encoder_dim = encoder_dim
        self.n_per_dim = n_per_dim
        self.skip_layers = skip_layers

        # Skip projections: token (B, 64, 1024) → reshape 3D → upsample/conv to target res
        # skip1: layer 18 → 4³ → 8³, 256 ch
        # skip2: layer 12 → 4³ → 16³, 128 ch
        # skip3: layer 6  → 4³ → 32³, 64 ch
        self.skip1 = self._skip_block(encoder_dim, 256, scale=2)
        self.skip2 = self._skip_block(encoder_dim, 128, scale=4)
        self.skip3 = self._skip_block(encoder_dim,  64, scale=8)

        # Bottleneck upsamples
        # 4³ → 8³ → 16³ → 32³ → 64³
        self.up1 = nn.ConvTranspose3d(encoder_dim, 512, kernel_size=2, stride=2)
        self.merge1 = self._merge_block(512 + 256, 512)

        self.up2 = nn.ConvTranspose3d(512, 256, kernel_size=2, stride=2)
        self.merge2 = self._merge_block(256 + 128, 256)

        self.up3 = nn.ConvTranspose3d(256, 128, kernel_size=2, stride=2)
        self.merge3 = self._merge_block(128 + 64, 128)

        self.up4 = nn.ConvTranspose3d(128, 64, kernel_size=2, stride=2)
        self.final = nn.Sequential(
            nn.Conv3d(64, 64, 3, padding=1),
            nn.InstanceNorm3d(64), nn.GELU(),
            nn.Conv3d(64, 1, 1),
        )

    def _skip_block(self, in_ch, out_ch, scale):
        return nn.Sequential(
            nn.ConvTranspose3d(in_ch, out_ch, kernel_size=scale, stride=scale),
            nn.InstanceNorm3d(out_ch), nn.GELU(),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch), nn.GELU(),
        )

    def _merge_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch), nn.GELU(),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch), nn.GELU(),
        )

    def _tokens_to_3d(self, tokens):
        """(B, 64, 1024) → (B, 1024, 4, 4, 4)"""
        B = tokens.shape[0]
        x = tokens.view(B, self.n_per_dim, self.n_per_dim, self.n_per_dim, self.encoder_dim)
        return x.permute(0, 4, 1, 2, 3).contiguous()

    def forward(self, tokens, hidden_states):
        # Skip features (3 layers, 1-indexed → 0-indexed)
        # hidden_states[i] = output of layer i+1
        skip1 = self._tokens_to_3d(hidden_states[self.skip_layers[2] - 1])  # layer 18
        skip2 = self._tokens_to_3d(hidden_states[self.skip_layers[1] - 1])  # layer 12
        skip3 = self._tokens_to_3d(hidden_states[self.skip_layers[0] - 1])  # layer 6

        # Project skips to their target resolutions
        skip1_8  = self.skip1(skip1)   # (B, 256, 8, 8, 8)
        skip2_16 = self.skip2(skip2)   # (B, 128, 16, 16, 16)
        skip3_32 = self.skip3(skip3)   # (B,  64, 32, 32, 32)

        # Bottleneck (final encoder output)
        x = self._tokens_to_3d(tokens)  # (B, 1024, 4, 4, 4)

        # Up 1: 4³ → 8³
        x = self.up1(x)                              # (B, 512, 8, 8, 8)
        x = torch.cat([x, skip1_8], dim=1)
        x = self.merge1(x)                           # (B, 512, 8, 8, 8)

        # Up 2: 8³ → 16³
        x = self.up2(x)                              # (B, 256, 16, 16, 16)
        x = torch.cat([x, skip2_16], dim=1)
        x = self.merge2(x)                           # (B, 256, 16, 16, 16)

        # Up 3: 16³ → 32³
        x = self.up3(x)                              # (B, 128, 32, 32, 32)
        x = torch.cat([x, skip3_32], dim=1)
        x = self.merge3(x)                           # (B, 128, 32, 32, 32)

        # Up 4: 32³ → 64³
        x = self.up4(x)                              # (B, 64, 64, 64, 64)
        x = self.final(x)                            # (B, 1, 64, 64, 64)

        return x

print('UNETRLiteDecoder tanımlandı ✅')

UNETRLiteDecoder tanımlandı ✅


## 5. Concept Bottleneck Multi-task Model

In [5]:
class CBMMultiTaskModel(nn.Module):
    """
    Concept Bottleneck Multi-task Model.

    Forward dict output:
      detection:     (B, 2)
      concepts:      (B, 8)
      malignancy:    (B, 2)   ← Bottleneck: concept'lerden lineer
      segmentation:  (B, 1, 64, 64, 64)
    """

    def __init__(self,
                 roi_size=(64, 64, 64),
                 patch_size=(16, 16, 16),
                 hidden_size=1024,
                 mlp_dim=4096,
                 num_layers=24,
                 num_heads=16,
                 n_concepts=8,
                 skip_layers=(6, 12, 18),
                 head_hidden=256,
                 head_dropout=0.1):
        super().__init__()
        self.n_concepts = n_concepts

        # 1) Pretrained ViT-L encoder
        self.encoder = ViT(
            in_channels=1,
            img_size=roi_size,
            patch_size=patch_size,
            hidden_size=hidden_size,
            mlp_dim=mlp_dim,
            num_layers=num_layers,
            num_heads=num_heads,
            classification=False,
        )

        # 2) Detection head: global pool → MLP → binary
        self.detection_head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, head_hidden),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_hidden, 2),
        )

        # 3) Concept head: global pool → MLP → 8 regression outputs
        self.concept_head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, head_hidden),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_hidden, n_concepts),
        )

        # 4) Concept Bottleneck: malignancy = Linear(concepts → 2)
        #    Bu lineer projeksiyon klinik açıklanabilirliği sağlar.
        self.malignancy_head = nn.Linear(n_concepts, 2)

        # 5) Segmentation decoder
        self.seg_decoder = UNETRLiteDecoder(
            encoder_dim=hidden_size,
            n_per_dim=roi_size[0] // patch_size[0],
            skip_layers=skip_layers,
        )

    def forward(self, x):
        # x: (B, 1, 64, 64, 64)
        tokens, hidden_states = self.encoder(x)
        # tokens: (B, 64, 1024)
        # hidden_states: list of 24 tensors, her biri (B, 64, 1024)

        # Global feature: mean pool over token dim
        global_feat = tokens.mean(dim=1)  # (B, 1024)

        # Detection
        det_logits = self.detection_head(global_feat)  # (B, 2)

        # Concepts
        concept_preds = self.concept_head(global_feat)  # (B, 8)

        # Malignancy from concepts (BOTTLENECK!)
        mal_logits = self.malignancy_head(concept_preds)  # (B, 2)

        # Segmentation
        seg_logits = self.seg_decoder(tokens, hidden_states)  # (B, 1, 64, 64, 64)

        return {
            'detection':    det_logits,
            'concepts':     concept_preds,
            'malignancy':   mal_logits,
            'segmentation': seg_logits,
        }

print('CBMMultiTaskModel tanımlandı ✅')

CBMMultiTaskModel tanımlandı ✅


## 6. Modeli oluştur ve DAPT encoder weights'i yükle

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Model oluşturuluyor...')
model = CBMMultiTaskModel(
    roi_size=ROI_SIZE,
    patch_size=PATCH_SIZE,
    hidden_size=HIDDEN_SIZE,
    mlp_dim=MLP_DIM,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    n_concepts=N_CONCEPTS,
    skip_layers=SKIP_LAYERS,
).to(device)
print(f'Model device: {device}')

print(f'\nDAPT checkpoint yükleniyor: {os.path.basename(DAPT_CKPT)}')
if not os.path.exists(DAPT_CKPT):
    print(f'⚠️ Checkpoint bulunamadı! Path kontrol et.')
    print(f'   Beklenen: {DAPT_CKPT}')
    print(f'   Drive\'da olanlar:')
    for f in os.listdir(os.path.dirname(DAPT_CKPT)):
        print(f'     {f}')
else:
    # model_only.pth → sadece state_dict
    # full_state.pth → optimizer/scheduler dahil dict
    ckpt = torch.load(DAPT_CKPT, map_location=device)

    # Hangi format?
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        encoder_state = ckpt['model_state_dict']
        print(f'  full_state ckpt — model_state_dict çıkarıldı')
    else:
        encoder_state = ckpt
        print(f'  model_only ckpt')

    # Encoder'a yükle (strict=False — head katmanlari henüz yok)
    missing, unexpected = model.encoder.load_state_dict(encoder_state, strict=False)
    print(f'  Eksik anahtarlar (head katmanlarina ait, beklenir): {len(missing)}')
    print(f'  Beklenmedik anahtarlar (decoder ise normal): {len(unexpected)}')
    if len(missing) > 0 and len(missing) < 5:
        print(f'    Missing: {missing}')
    if len(unexpected) > 0 and len(unexpected) < 5:
        print(f'    Unexpected: {unexpected}')
    print(f'\n✅ Encoder weights yüklendi')

Model oluşturuluyor...
Model device: cuda

DAPT checkpoint yükleniyor: pulmomae_dapt_epoch3_20260525-210449_model_only.pth
  model_only ckpt
  Eksik anahtarlar (head katmanlarina ait, beklenir): 0
  Beklenmedik anahtarlar (decoder ise normal): 0

✅ Encoder weights yüklendi


## 7. Parameter count

In [7]:
def count_params(module):
    return sum(p.numel() for p in module.parameters())

def count_trainable(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

print('=== PARAMETRE SAYIMI ===')
print(f'  Encoder:               {count_params(model.encoder)/1e6:>7.1f} M')
print(f'  Detection head:        {count_params(model.detection_head)/1e6:>7.2f} M')
print(f'  Concept head:          {count_params(model.concept_head)/1e6:>7.2f} M')
print(f'  Malignancy (Linear):   {count_params(model.malignancy_head)/1e6:>7.4f} M  ← Bottleneck')
print(f'  Seg decoder:           {count_params(model.seg_decoder)/1e6:>7.1f} M')
print(f'  ─────────────────────────────')
print(f'  TOPLAM:                {count_params(model)/1e6:>7.1f} M')

print(f'\nTrainable: {count_trainable(model)/1e6:.1f} M')
print(f'Frozen:    {(count_params(model) - count_trainable(model))/1e6:.1f} M')

=== PARAMETRE SAYIMI ===
  Encoder:                 407.2 M
  Detection head:           0.26 M
  Concept head:             0.27 M
  Malignancy (Linear):    0.0000 M  ← Bottleneck
  Seg decoder:              75.3 M
  ─────────────────────────────
  TOPLAM:                  483.0 M

Trainable: 483.0 M
Frozen:    0.0 M


## 8. Forward pass smoke test

Random input → output shapes doğru mu?

In [8]:
print('Forward smoke test...')

model.eval()
B = 2
dummy_input = torch.randn(B, 1, *ROI_SIZE, device=device)
print(f'Input: {dummy_input.shape}')

with torch.no_grad():
    out = model(dummy_input)

print(f'\nOutputs:')
expected = {
    'detection':    (B, 2),
    'concepts':     (B, N_CONCEPTS),
    'malignancy':   (B, 2),
    'segmentation': (B, 1, *ROI_SIZE),
}
all_ok = True
for k, v in out.items():
    ok = '✅' if tuple(v.shape) == expected[k] else '❌'
    if tuple(v.shape) != expected[k]:
        all_ok = False
    print(f'  {ok} {k:14s}: {tuple(v.shape)}  expected {expected[k]}')

if all_ok:
    print(f'\n✅ Tüm shape\'ler doğru')

# Çıktı dağılımları (init kontrol için)
print(f'\nÇıktı istatistikleri (eval mode, random input):')
for k, v in out.items():
    print(f'  {k:14s}: range=[{v.min().item():.3f}, {v.max().item():.3f}], '
          f'mean={v.mean().item():.3f}, std={v.std().item():.3f}')

Forward smoke test...
Input: torch.Size([2, 1, 64, 64, 64])

Outputs:
  ✅ detection     : (2, 2)  expected (2, 2)
  ✅ concepts      : (2, 8)  expected (2, 8)
  ✅ malignancy    : (2, 2)  expected (2, 2)
  ✅ segmentation  : (2, 1, 64, 64, 64)  expected (2, 1, 64, 64, 64)

✅ Tüm shape'ler doğru

Çıktı istatistikleri (eval mode, random input):
  detection     : range=[-0.134, 0.157], mean=0.011, std=0.168
  concepts      : range=[-0.408, 0.266], mean=-0.024, std=0.218
  malignancy    : range=[-0.339, 0.308], mean=-0.017, std=0.372
  segmentation  : range=[-1.601, 1.289], mean=-0.169, std=0.332


## 9. Multi-task loss function

In [9]:
def multitask_loss(outputs, batch, weights=None, eps=1e-7):
    """
    Multi-task loss:
      L = w_det * L_det + w_seg * L_seg + w_mal * L_mal + w_concept * L_concept

    Args:
      outputs: dict from model.forward
      batch: dict with keys 'detection', 'malignancy', 'mask', 'seg_loss_mask', 'concepts'
      weights: dict, default {'detection': 1.0, ...}

    Returns:
      losses: dict of per-task + total loss
    """
    if weights is None:
        weights = {'detection': 1.0, 'segmentation': 1.0, 'malignancy': 1.0, 'concepts': 1.0}

    losses = {}

    # === Detection: BCE (always active) ===
    det_logits = outputs['detection']
    det_target = batch['detection']  # (B,) long, 0/1
    losses['detection'] = F.cross_entropy(det_logits, det_target)

    # === Malignancy: BCE with ignore_index=-1 ===
    mal_logits = outputs['malignancy']
    mal_target = batch['malignancy']  # (B,) long, -1 ignored
    # Eğer batch'te hiç valid malignancy yoksa loss = 0
    if (mal_target != -1).any():
        losses['malignancy'] = F.cross_entropy(mal_logits, mal_target, ignore_index=-1)
    else:
        losses['malignancy'] = torch.tensor(0.0, device=mal_logits.device)

    # === Segmentation: Dice + BCE, masked by seg_loss_mask ===
    seg_logits = outputs['segmentation']  # (B, 1, 64, 64, 64)
    seg_target = batch['mask'].float()    # (B, 1, 64, 64, 64)
    seg_mask = batch['seg_loss_mask'].view(-1, 1, 1, 1, 1).float()  # (B, 1, 1, 1, 1)

    # BCE per voxel, masked
    bce_per_voxel = F.binary_cross_entropy_with_logits(seg_logits, seg_target, reduction='none')
    bce = (bce_per_voxel * seg_mask).sum() / (seg_mask.sum() * 64 * 64 * 64 + eps)

    # Dice loss (per-sample, then mask)
    seg_probs = torch.sigmoid(seg_logits)
    dims = (2, 3, 4)
    inter = (seg_probs * seg_target).sum(dim=dims).sum(dim=1)        # (B,)
    union = seg_probs.sum(dim=dims).sum(dim=1) + seg_target.sum(dim=dims).sum(dim=1)  # (B,)
    dice_per_sample = 1.0 - (2.0 * inter + 1.0) / (union + 1.0)       # (B,)
    seg_mask_per_sample = seg_mask.view(-1)                          # (B,)
    if seg_mask_per_sample.sum() > 0:
        dice_loss = (dice_per_sample * seg_mask_per_sample).sum() / (seg_mask_per_sample.sum() + eps)
    else:
        dice_loss = torch.tensor(0.0, device=seg_logits.device)

    losses['segmentation'] = bce + dice_loss

    # === Concept: MSE, masked when concept = -1 ===
    concept_preds = outputs['concepts']
    concept_target = batch['concepts'].float()  # (B, 8)
    # Concept geçerli mi? -1.0 default değer
    concept_mask = (concept_target[:, 0] != -1.0).float().unsqueeze(1)  # (B, 1)

    mse_per = F.mse_loss(concept_preds, concept_target, reduction='none')  # (B, 8)
    if concept_mask.sum() > 0:
        losses['concepts'] = (mse_per * concept_mask).sum() / (concept_mask.sum() * concept_preds.shape[1] + eps)
    else:
        losses['concepts'] = torch.tensor(0.0, device=concept_preds.device)

    # === Weighted total ===
    total = (weights['detection']    * losses['detection']
           + weights['segmentation'] * losses['segmentation']
           + weights['malignancy']   * losses['malignancy']
           + weights['concepts']     * losses['concepts'])
    losses['total'] = total

    return losses

print('multitask_loss tanımlandı ✅')

multitask_loss tanımlandı ✅


## 10. Loss smoke test — dummy batch

In [10]:
print('Loss smoke test (dummy batch)...')
model.train()

B = 4
dummy_batch = {
    'patch':         torch.randn(B, 1, *ROI_SIZE, device=device),
    'mask':          (torch.rand(B, 1, *ROI_SIZE, device=device) > 0.95).float(),  # %5 voxel mask
    'detection':     torch.tensor([1, 1, 0, 0], dtype=torch.long, device=device),
    'malignancy':    torch.tensor([1, 0, -1, -1], dtype=torch.long, device=device),
    'seg_loss_mask': torch.tensor([1.0, 1.0, 0.0, 0.0], device=device),
    'concepts':      torch.tensor([
        [4.2, 1.0, 6.0, 3.5, 2.1, 3.8, 4.0, 4.8],
        [2.0, 1.0, 6.0, 4.2, 4.5, 1.5, 1.2, 4.9],
        [-1, -1, -1, -1, -1, -1, -1, -1],
        [-1, -1, -1, -1, -1, -1, -1, -1],
    ], device=device),
}

outputs = model(dummy_batch['patch'])
losses = multitask_loss(outputs, dummy_batch)

print(f'\nLoss değerleri (initial, random head katmanlari):')
for k, v in losses.items():
    print(f'  {k:14s}: {v.item():.4f}')

# Backward smoke test
print('\nBackward pass smoke test...')
losses['total'].backward()
print('  ✅ Gradient hesaplari yapildi (RuntimeError yok)')

# Gradient varlığı kontrolü
print('\nGradient sağlık check:')
has_grad = {}
for name, module in [('encoder', model.encoder),
                      ('det_head', model.detection_head),
                      ('concept_head', model.concept_head),
                      ('mal_head', model.malignancy_head),
                      ('seg_decoder', model.seg_decoder)]:
    grads = [p.grad for p in module.parameters() if p.grad is not None]
    if grads:
        g_norm = torch.cat([g.flatten() for g in grads]).norm().item()
        has_grad[name] = g_norm
        print(f'  {name:15s}: grad norm = {g_norm:.4f}')
    else:
        print(f'  {name:15s}: ⚠️ NO GRADIENT')

model.zero_grad()

Loss smoke test (dummy batch)...

Loss değerleri (initial, random head katmanlari):
  detection     : 0.6844
  malignancy    : 0.7706
  segmentation  : 1.5440
  concepts      : 14.3171
  total         : 17.3161

Backward pass smoke test...
  ✅ Gradient hesaplari yapildi (RuntimeError yok)

Gradient sağlık check:
  encoder        : grad norm = 5.2779
  det_head       : grad norm = 2.1074
  concept_head   : grad norm = 34.7505
  mal_head       : grad norm = 0.3113
  seg_decoder    : grad norm = 1.0389


## 11. Memory profile — kaç batch sığar?

Farklı batch boyutlarında forward+backward VRAM kullanımı.

In [11]:
def measure_vram(batch_size):
    """Belirli batch'te forward+backward VRAM peak'i ölç."""
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    try:
        x = torch.randn(batch_size, 1, *ROI_SIZE, device=device)
        batch = {
            'patch': x,
            'mask': torch.zeros(batch_size, 1, *ROI_SIZE, device=device),
            'detection':  torch.zeros(batch_size, dtype=torch.long, device=device),
            'malignancy': torch.full((batch_size,), -1, dtype=torch.long, device=device),
            'seg_loss_mask': torch.zeros(batch_size, device=device),
            'concepts': torch.full((batch_size, N_CONCEPTS), -1.0, device=device),
        }
        # Set 1 sample as valid for non-zero seg loss
        batch['detection'][0] = 1
        batch['seg_loss_mask'][0] = 1.0
        batch['malignancy'][0] = 1
        batch['concepts'][0] = torch.tensor([4.0]*N_CONCEPTS, device=device)

        model.train()
        outputs = model(batch['patch'])
        losses = multitask_loss(outputs, batch)
        losses['total'].backward()

        peak = torch.cuda.max_memory_allocated() / 1e9

        del x, batch, outputs, losses
        model.zero_grad()
        torch.cuda.empty_cache()

        return peak, True
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return None, False
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            torch.cuda.empty_cache()
            return None, False
        raise

if torch.cuda.is_available():
    print('VRAM kullanımı (forward + backward):')
    print(f'{"Batch":>6}  {"VRAM (GB)":>10}  {"Durum":>10}')
    for B in [1, 2, 4, 8, 16]:
        peak, ok = measure_vram(B)
        status = '✅ OK' if ok else '❌ OOM'
        peak_str = f'{peak:.2f}' if peak else 'N/A'
        print(f'  {B:>3}    {peak_str:>10}    {status:>10}')

    print(f'\nGPU: {torch.cuda.get_device_name(0)}, '
          f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('CUDA yok, VRAM profili atlanıyor.')

VRAM kullanımı (forward + backward):
 Batch   VRAM (GB)       Durum
    1          5.62          ✅ OK
    2          5.63          ✅ OK
    4          6.90          ✅ OK
    8          9.84          ✅ OK
   16         15.72          ✅ OK

GPU: NVIDIA A100-SXM4-80GB, 85.1 GB


## 12. Özet

In [12]:
total_params = count_params(model) / 1e6
trainable = count_trainable(model) / 1e6

print(f'''
============================================================
NOTEBOOK 5 — MODEL MİMARİSİ ÖZET
============================================================

🧠 Model: Concept Bottleneck Multi-task Model
  Encoder:    ViT-Large (pretrained DAPT epoch 3)
  4 head:     detection, concepts, malignancy (CBM), segmentation
  Total:      {total_params:.1f} M parameter
  Trainable:  {trainable:.1f} M

🎯 Konsept bottleneck garanti:
  Malignancy = Linear(concepts → 2)
  Model malignancy kararı için MECBUREN 8 klinik
  kavram üzerinden geçer → açıklanabilir CADx

📐 Output şekilleri:
  detection:     (B, 2)
  concepts:      (B, 8)
  malignancy:    (B, 2)
  segmentation:  (B, 1, 64, 64, 64)

🎯 Sıradaki: Notebook 6 — Fine-tune training
  • H5 Dataset class (concept dahil, runtime negatif sampling)
  • Multi-task training loop (AdamW + cosine schedule)
  • Drive checkpoint + telegram, FROC/AUC/Dice metrikleri
  • Memory-aware (batch + gradient accum + bf16)
''')


NOTEBOOK 5 — MODEL MİMARİSİ ÖZET

🧠 Model: Concept Bottleneck Multi-task Model
  Encoder:    ViT-Large (pretrained DAPT epoch 3)
  4 head:     detection, concepts, malignancy (CBM), segmentation
  Total:      483.0 M parameter
  Trainable:  483.0 M

🎯 Konsept bottleneck garanti:
  Malignancy = Linear(concepts → 2)
  Model malignancy kararı için MECBUREN 8 klinik
  kavram üzerinden geçer → açıklanabilir CADx

📐 Output şekilleri:
  detection:     (B, 2)
  concepts:      (B, 8)
  malignancy:    (B, 2)
  segmentation:  (B, 1, 64, 64, 64)

🎯 Sıradaki: Notebook 6 — Fine-tune training
  • H5 Dataset class (concept dahil, runtime negatif sampling)
  • Multi-task training loop (AdamW + cosine schedule)
  • Drive checkpoint + telegram, FROC/AUC/Dice metrikleri
  • Memory-aware (batch + gradient accum + bf16)

